# Domain Adaptation: DANN (Domain Adversarial Neural Networks) - Hyperparameter Tuning

Goals:
- optimise the DANN adversarial weight λ
- compare with MMD and CORAL
- test all 3 properties

DANN Loss:
- L_task = Task prediction loss (Huber)
- L_adv = Domain classifier loss (BCEWithLogits)
- Total Loss = L_task - λ * L_adv (gradient reversal)

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import spearmanr
from sklearn.metrics import r2_score
import json
import warnings
warnings.filterwarnings('ignore')

ROOT = r'..'
DATA_DIR = os.path.join(ROOT, 'data', 'cv_datasets')
FEAT_DIR = os.path.join(ROOT, 'features')
RES_DIR = os.path.join(ROOT, 'results', 'dann_tuning')
os.makedirs(RES_DIR, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PROPERTIES = ['clearance', 'half_life', 'fu']
N_FOLDS = 5

# Hyperparameter Grid
DANN_WEIGHTS = [0.01, 0.05, 0.1, 0.3, 0.5, 1.0]

print("="*100)
print("Domain Adaptation: DANN (Domain Adversarial Neural Networks) - Hyperparameter Tuning")
print("="*100)
print(f"Device: {DEVICE}")
print(f"Properties: {PROPERTIES}")
print(f"N Folds: {N_FOLDS}")
print(f"DANN Adversarial Weights to test: {DANN_WEIGHTS}\n")

# Shared training utilities: inner train/val split + val-based early stopping (test used only for final evaluation)
sys.path.insert(0, os.path.join(ROOT, '01_model'))
from train_utils import split_train_val, fit_early_stopping, target_smiles


Domain Adaptation: DANN (Domain Adversarial Neural Networks) - Hyperparameter Tuning
Device: cpu
Properties: ['clearance', 'half_life', 'fu']
N Folds: 5
DANN Adversarial Weights to test: [0.01, 0.05, 0.1, 0.3, 0.5, 1.0]



In [2]:
print("Loading Graphormer embeddings...\n")

emb_path = os.path.join(FEAT_DIR, 'graphormer_cv_embeddings.npz')
emb_data = np.load(emb_path, allow_pickle=True)
emb_keys = emb_data['keys']
emb_X = emb_data['X']

smiles_to_emb = {smi: emb_X[i] for i, smi in enumerate(emb_keys)}

print(f"✓ Loaded {len(smiles_to_emb)} embeddings")
print(f"  Dim: {emb_X.shape[1]}\n")

Loading Graphormer embeddings...

✓ Loaded 58304 embeddings
  Dim: 768



In [3]:
class ReverseLayerF(torch.autograd.Function):
    """Gradient reversal layer"""
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.alpha, None

class SharedEncoder(nn.Module):
    def __init__(self, d_in=768, d_hidden=256, dropout=0.2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_in, d_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_hidden, d_hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.d_out = d_hidden // 2

    def forward(self, x):
        return self.encoder(x)

class TaskHead(nn.Module):
    def __init__(self, d_in, d_hidden=128):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(d_in, d_hidden),
            nn.ReLU(),
            nn.Linear(d_hidden, 1),
        )

    def forward(self, x):
        return self.head(x).squeeze(-1)

class DomainClassifier(nn.Module):
    """Binary domain classifier for adversarial training"""
    def __init__(self, d_in, d_hidden=64):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(d_in, d_hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(d_hidden, 1),
        )

    def forward(self, x):
        return self.classifier(x).squeeze(-1)

class DANNModel(nn.Module):
    """DANN: Domain Adversarial Neural Networks"""
    def __init__(self, d_in=768):
        super().__init__()
        self.encoder = SharedEncoder(d_in)
        self.task_head = TaskHead(self.encoder.d_out)
        self.domain_classifier = DomainClassifier(self.encoder.d_out)

    def forward(self, x, alpha=0.0):
        features = self.encoder(x)
        
        # Task prediction
        task_out = self.task_head(features)
        
        # Domain classification (with gradient reversal)
        if alpha > 0:
            reversed_features = ReverseLayerF.apply(features, alpha)
            domain_out = self.domain_classifier(reversed_features)
        else:
            domain_out = self.domain_classifier(features)
        
        return task_out, domain_out, features

print("✓ DANN model defined")

✓ DANN model defined


In [4]:
def get_smiles_col(df):
    if 'smiles' in df.columns:
        return 'smiles'
    elif 'Drug' in df.columns:
        return 'Drug'
    else:
        return None

def get_value_col(df):
    if 'value' in df.columns:
        return 'value'
    elif 'Y' in df.columns:
        return 'Y'
    else:
        return None

def load_source(prop, fold_idx):
    fold_dir = os.path.join(DATA_DIR, f'fold_{fold_idx}')
    csv_path = os.path.join(fold_dir, f'{prop}_source.csv')
    
    if not os.path.exists(csv_path):
        return None, None
    
    df = pd.read_csv(csv_path, low_memory=False)
    smi_col = get_smiles_col(df)
    val_col = get_value_col(df)
    
    if smi_col is None or val_col is None:
        return None, None
    
    smiles = df[smi_col].astype(str).tolist()
    
    X_list = []
    y_list = []
    for smi, y in zip(smiles, df[val_col].values):
        if smi in smiles_to_emb:
            X_list.append(smiles_to_emb[smi])
            y_list.append(y)
    
    X = np.stack(X_list).astype('float32') if X_list else np.array([])
    y = np.array(y_list, dtype='float32') if y_list else np.array([])
    return X, y

def load_target(prop, fold_idx, split):
    fold_dir = os.path.join(DATA_DIR, f'fold_{fold_idx}')
    csv_path = os.path.join(fold_dir, f'{prop}_target_{split}.csv')
    
    if not os.path.exists(csv_path):
        return None, None
    
    df = pd.read_csv(csv_path, low_memory=False)
    smi_col = get_smiles_col(df)
    val_col = get_value_col(df)
    
    if smi_col is None or val_col is None:
        return None, None
    
    smiles = df[smi_col].astype(str).tolist()
    
    X_list = []
    y_list = []
    for smi, y in zip(smiles, df[val_col].values):
        if smi in smiles_to_emb:
            X_list.append(smiles_to_emb[smi])
            y_list.append(y)
    
    X = np.stack(X_list).astype('float32') if X_list else np.array([])
    y = np.array(y_list, dtype='float32') if y_list else np.array([])
    return X, y

print("✓ Data loaders defined")

✓ Data loaders defined


In [5]:
def train_epoch_dann(model, X_src, y_src, X_tgt, y_tgt, optimizer, criterion, dann_w=0.1, device='cpu'):
    """Train DANN model for one epoch"""
    model.train()
    
    X_src = np.asarray(X_src, dtype='float32')
    y_src = np.asarray(y_src, dtype='float32')
    X_tgt = np.asarray(X_tgt, dtype='float32')
    y_tgt = np.asarray(y_tgt, dtype='float32')
    
    # Epochs count target batches; the source is randomly sampled from the whole set each batch (independent of source size)
    n = len(X_tgt)
    perm = np.random.permutation(n)
    
    task_loss_sum = 0.0
    dann_loss_sum = 0.0
    n_batches = 0

    for i in range(0, n, 32):
        idx = perm[i:i + 32]
        idx_s = np.random.randint(0, len(X_src), len(idx))
        
        x_src = torch.tensor(X_src[idx_s], device=device, dtype=torch.float32)
        y_src_batch = torch.tensor(y_src[idx_s], device=device, dtype=torch.float32)
        x_tgt = torch.tensor(X_tgt[idx], device=device, dtype=torch.float32)
        y_tgt_batch = torch.tensor(y_tgt[idx], device=device, dtype=torch.float32)
        
        optimizer.zero_grad()
        
        # Forward pass
        y_pred_tgt, domain_pred, _ = model(x_tgt, alpha=dann_w)
        _, domain_src, _ = model(x_src, alpha=dann_w)
        
        # Task loss (on target)
        task_loss = criterion(y_pred_tgt, y_tgt_batch)
        
        # Domain adversarial loss
        domain_labels_src = torch.zeros(domain_src.shape[0], device=device)  # source = 0
        domain_labels_tgt = torch.ones(domain_pred.shape[0], device=device)  # target = 1
        
        domain_loss_src = F.binary_cross_entropy_with_logits(domain_src, domain_labels_src)
        domain_loss_tgt = F.binary_cross_entropy_with_logits(domain_pred, domain_labels_tgt)
        domain_loss = domain_loss_src + domain_loss_tgt
        
        # Total loss (adversarial loss is subtracted due to gradient reversal)
        total_loss = task_loss + dann_w * domain_loss
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        
        task_loss_sum += task_loss.item()
        dann_loss_sum += domain_loss.item()
        n_batches += 1

    return task_loss_sum / max(n_batches, 1), dann_loss_sum / max(n_batches, 1)

@torch.no_grad()
def evaluate(model, X, y, device):
    model.eval()
    X = np.asarray(X, dtype='float32')
    y = np.asarray(y, dtype='float32')
    
    x = torch.tensor(X, device=device, dtype=torch.float32)
    y_pred, _, _ = model(x, alpha=0.0)  # No gradient reversal during inference
    y_pred = y_pred.cpu().numpy()
    
    # Handle NaN/Inf values
    y_pred = np.nan_to_num(y_pred, nan=0.0, posinf=1e6, neginf=-1e6)
    
    r = np.corrcoef(y, y_pred)[0, 1]
    rho, _ = spearmanr(y, y_pred)
    r2 = r2_score(y, y_pred)
    rmse = np.sqrt(np.mean((y - y_pred) ** 2))
    mae = np.mean(np.abs(y - y_pred))
    mbe = np.mean(y_pred - y)
    
    return {'r': float(r), 'rho': float(rho), 'r2': float(r2), 'rmse': float(rmse), 'mae': float(mae), 'mbe': float(mbe)}

print("✓ Training functions defined")

✓ Training functions defined


In [6]:
# GRID SEARCH: Test all DANN weights
print("\n" + "="*100)
print("HYPERPARAMETER TUNING: DANN Adversarial Weight Grid Search")
print("="*100 + "\n")

tuning_results = {}  # {prop: {dann_w: results_dict}}

for prop in PROPERTIES:
    print(f"\n{'='*80}")
    print(f"Property: {prop.upper()}")
    print(f"{'='*80}")
    
    tuning_results[prop] = {}
    
    for dann_w in DANN_WEIGHTS:
        print(f"\n  Testing DANN Weight = {dann_w}")
        fold_results = []
        
        for fold_idx in range(N_FOLDS):
            X_src, y_src = load_source(prop, fold_idx)
            X_tgt_train, y_tgt_train = load_target(prop, fold_idx, 'train')
            X_tgt_test, y_tgt_test = load_target(prop, fold_idx, 'test')
            
            if X_src is None or len(X_src) == 0:
                continue
            
            # The source is not truncated; everything is used (randomly sampled each batch during training)
            X_src = np.asarray(X_src, dtype='float32')
            y_src = np.asarray(y_src, dtype='float32')
            # target train → scaffold-based inner train/val (early stopping and λ selection on val, test used only for final evaluation)
            X_tgt_train, y_tgt_train, X_tgt_val, y_tgt_val = split_train_val(
                np.asarray(X_tgt_train, dtype='float32'), np.asarray(y_tgt_train, dtype='float32'), fold_idx,
                smiles=target_smiles(DATA_DIR, prop, fold_idx, 'train', smiles_to_emb))
            X_tgt_test = np.asarray(X_tgt_test, dtype='float32')
            y_tgt_test = np.asarray(y_tgt_test, dtype='float32')
            
            torch.manual_seed(fold_idx)
            np.random.seed(fold_idx)
            
            model = DANNModel().to(DEVICE)
            optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
            criterion = nn.HuberLoss(delta=1.0)
            
            def run_epoch():
                train_epoch_dann(model, X_src, y_src, X_tgt_train, y_tgt_train, optimizer, criterion, dann_w, DEVICE)
            
            best_epoch, _ = fit_early_stopping(
                model, run_epoch, lambda m, X, y: evaluate(m, X, y, DEVICE), X_tgt_val, y_tgt_val)
            val_metrics = evaluate(model, X_tgt_val, y_tgt_val, DEVICE)
            metrics = evaluate(model, X_tgt_test, y_tgt_test, DEVICE)
            metrics.update({f'val_{k}': v for k, v in val_metrics.items()})
            metrics['best_epoch'] = best_epoch
            fold_results.append(metrics)
        
        if fold_results:
            results_df = pd.DataFrame(fold_results)
            tuning_results[prop][dann_w] = {
                'rho_mean': float(results_df['rho'].mean()),
                'rho_std': float(results_df['rho'].std()),
                'r2_mean': float(results_df['r2'].mean()),
                'r2_std': float(results_df['r2'].std()),
                'rmse_mean': float(results_df['rmse'].mean()),
                'rmse_std': float(results_df['rmse'].std()),
                'val_rho_mean': float(results_df['val_rho'].mean()),
                'val_r2_mean': float(results_df['val_r2'].mean()),
            }
            
            print(f"    ρ: {results_df['rho'].mean():.4f}±{results_df['rho'].std():.4f} | R²: {results_df['r2'].mean():.4f}±{results_df['r2'].std():.4f}")
            
            # Save fold results
            results_df.to_csv(os.path.join(RES_DIR, f'{prop}_dann{dann_w}.csv'), index=False)


HYPERPARAMETER TUNING: DANN Adversarial Weight Grid Search


Property: CLEARANCE

  Testing DANN Weight = 0.01
    ρ: 0.3354±0.0612 | R²: 0.0661±0.0616

  Testing DANN Weight = 0.05
    ρ: 0.3700±0.0463 | R²: 0.1136±0.0495

  Testing DANN Weight = 0.1
    ρ: 0.3566±0.0471 | R²: 0.0933±0.0576

  Testing DANN Weight = 0.3
    ρ: 0.3706±0.0469 | R²: 0.0913±0.0428

  Testing DANN Weight = 0.5
    ρ: 0.3579±0.0295 | R²: 0.1021±0.0168

  Testing DANN Weight = 1.0
    ρ: 0.3139±0.0972 | R²: 0.0748±0.0392

Property: HALF_LIFE

  Testing DANN Weight = 0.01
    ρ: 0.3913±0.0461 | R²: 0.0701±0.1175

  Testing DANN Weight = 0.05
    ρ: 0.4044±0.0504 | R²: 0.0672±0.1369

  Testing DANN Weight = 0.1
    ρ: 0.3937±0.0510 | R²: 0.0884±0.0745

  Testing DANN Weight = 0.3
    ρ: 0.4040±0.0736 | R²: 0.0976±0.1092

  Testing DANN Weight = 0.5
    ρ: 0.3780±0.0463 | R²: 0.0636±0.1486

  Testing DANN Weight = 1.0
    ρ: 0.3925±0.0688 | R²: 0.0958±0.0632

Property: FU

  Testing DANN Weight = 0.01
    ρ: 0.

In [7]:
# Summary Report
print(f"\n\n{'='*100}")
print("DANN TUNING RESULTS SUMMARY")
print(f"{'='*100}\n")

for prop in PROPERTIES:
    print(f"\n{prop.upper()}:")
    print(f"  {'DANN_W':<10} {'Spearman ρ':<20} {'R²':<20} {'RMSE':<20}")
    print(f"  {'-'*70}")
    
    best_rho = -1
    best_r2 = -1
    best_w_rho = None
    best_w_r2 = None
    
    for dann_w in DANN_WEIGHTS:
        if dann_w in tuning_results[prop]:
            res = tuning_results[prop][dann_w]
            rho_str = f"{res['rho_mean']:.4f}±{res['rho_std']:.4f}"
            r2_str = f"{res['r2_mean']:.4f}±{res['r2_std']:.4f}"
            rmse_str = f"{res['rmse_mean']:.4f}±{res['rmse_std']:.4f}"
            
            marker = ""
            if res['val_rho_mean'] > best_rho:
                best_rho = res['val_rho_mean']
                best_w_rho = dann_w
                marker += "(Best val ρ)"
            if res['val_r2_mean'] > best_r2:
                best_r2 = res['val_r2_mean']
                best_w_r2 = dann_w
                marker += "(Best val R²)"
            
            print(f"  {dann_w:<10.2f} {rho_str:<20} {r2_str:<20} {rmse_str:<20} {marker}")
    
    print(f"\n  Best DANN_W for ρ: {best_w_rho} ({best_rho:.4f})")
    print(f"  Best DANN_W for R²: {best_w_r2} ({best_r2:.4f})")



DANN TUNING RESULTS SUMMARY


CLEARANCE:
  DANN_W     Spearman ρ           R²                   RMSE                
  ----------------------------------------------------------------------
  0.01       0.3354±0.0612        0.0661±0.0616        0.9636±0.0968        (Best val ρ)(Best val R²)
  0.05       0.3700±0.0463        0.1136±0.0495        0.9401±0.1104        (Best val ρ)
  0.10       0.3566±0.0471        0.0933±0.0576        0.9489±0.0891        (Best val ρ)
  0.30       0.3706±0.0469        0.0913±0.0428        0.9511±0.0996        (Best val ρ)
  0.50       0.3579±0.0295        0.1021±0.0168        0.9452±0.0915        
  1.00       0.3139±0.0972        0.0748±0.0392        0.9601±0.1038        

  Best DANN_W for ρ: 0.3 (0.3896)
  Best DANN_W for R²: 0.01 (0.1535)

HALF_LIFE:
  DANN_W     Spearman ρ           R²                   RMSE                
  ----------------------------------------------------------------------
  0.01       0.3913±0.0461        0.0701±0.1175      

In [8]:
# Save tuning results
tuning_summary = {}
for prop in PROPERTIES:
    tuning_summary[prop] = {}
    for dann_w, results in tuning_results[prop].items():
        tuning_summary[prop][str(dann_w)] = {
            'rho_mean': float(results['rho_mean']),
            'rho_std': float(results['rho_std']),
            'r2_mean': float(results['r2_mean']),
            'r2_std': float(results['r2_std']),
            'rmse_mean': float(results['rmse_mean']),
            'rmse_std': float(results['rmse_std']),
            'val_rho_mean': float(results['val_rho_mean']),
            'val_r2_mean': float(results['val_r2_mean']),
        }

with open(os.path.join(RES_DIR, 'tuning_results.json'), 'w') as f:
    json.dump(tuning_summary, f, indent=2)

print(f"\n\n✓ DANN tuning results saved to {RES_DIR}")
print("\nDANN Hyperparameter Tuning Complete!")



✓ DANN tuning results saved to ..\results\dann_tuning

DANN Hyperparameter Tuning Complete!
